In [ ]:
import os 
from IPython.display import display
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from IPython.core.interactiveshell import InteractiveShell  
from ipywidgets import interact  
InteractiveShell.ast_node_interactivity = "all"   # type: ignore
import ipywidgets as widgets
from ipywidgets import interact 
from pandas import Index
from typing import Any, TypedDict, Type
import torch
from torchmetrics import Metric

from src.base.lit_module.components.metrics_standard import MyMultiClassAccuracy, MyLoss
from src.base.lit_module.components.metric_gch import RGCH_Metric, GCH_Metric

In [ ]:
import logging

log_file = Path(os.getcwd()) / "src/log_analysis.log"
logging.basicConfig(filename=log_file, level=logging.DEBUG)

In [ ]:
""" 
Required classes. 
"""

class MetricMetadata(TypedDict):
    epochs: np.ndarray
    steps: np.ndarray

class PerStepMetricInfo(TypedDict):
    values: np.ndarray
    save_path: Path
    header: list[str]

In [ ]:
outputs_dir = Path().resolve() / "outputs"
corr_store_names = dict(normed="GCH", raw="RGCH")

In [ ]:
""" 
Visualization functionality. 
"""

def viz_sep_col(metric: PerStepMetricInfo, save_fig: bool = True, save_derived_metric: bool = False) -> None:
    """
    For a metric object instance, visualize each col of `values` with its associated string in `header`.
    Internal output is figure with plots per column, header label combo.
    Optionally save each to compressed numpy file with flags (NOT recommended).
    Optionally save figure (Recommended). 
    TODO: But with what path, might as well maybe create separate instance separate from global df, not really. 
    """
    n_cols = metric["values"].shape[1]
    fig, axs = plt.subplots(1, n_cols, sharex=True, sharey=True, figsize=(4 * n_cols, 4))
    fig.subplots_adjust(wspace=0, hspace=0)

    for i, _ in enumerate(metric["header"]):
        col_vals = metric["values"][:,i]
        col_name = metric["header"][i]
        clean_header = col_name.replace('.', '_')
        col_path = Path(metric['save_path'].parent.resolve() / clean_header)
        axs[i].plot(
            col_vals, 
            # linestyle="dashed", 
            # color="black", 
            # marker="o", 
            label=clean_header
        )
        axs[i].legend()

        if save_derived_metric:
            if not Path.exists(col_path):
                col_path.mkdir(parents=True, exist_ok=True)
            np.savez_compressed(file=col_path.resolve(), derived_metric = col_vals)

    if save_fig: 
        save_path = Path(str(metric['save_path'].resolve()) + ".png")
        fig.savefig(save_path, format="png")

In [ ]:
def viz_all_col_sized(metric_df: pd.DataFrame, metadata_df: pd.DataFrame, unified_name: str = "Train Accuracy") -> None:
    """
    For a metric object instance, visualize all col of `values` with its associated string in `header`.
    Internal output is figure with 1 plot for all column , labelled by col name in Index.
    Optionally save as compressed numpy file with flags per viz (Recommended).
    Optionally save figure (Recommended) per viz. 
    """
    n_cols = len(metric_df.columns)
    fig, ax = plt.subplots(1, 1, sharex=True, sharey=True, figsize=(n_cols, n_cols))
    for col in metric_df.columns:
        # display(list(metric_df.index.get_level_values(1)))
        # display(metric_df.loc[:, col].describe())
        # display(metric_df.loc[:, col].max())
        line, = ax.plot(list(metric_df.index.to_numpy(dtype=np.int32)), list(metric_df.loc[:, col].values))
        line.set_label(col)
    ax.set_title(f"{unified_name} Plot", fontsize=200)

    ax.tick_params(axis='x', labelsize=70)
    ax.tick_params(axis='y', labelsize=70)

    ax.set_ylabel(unified_name, fontsize=120)
    ax.set_ylabel(unified_name, fontsize=120)

    ax.legend(fontsize=50, title_fontsize=90)
    plt.show()

In [ ]:
def viz_all_col_unsized(metric_df: pd.DataFrame, metadata_df: pd.DataFrame, unified_name: str = "Train Accuracy") -> None:
    """
    For a metric object instance, visualize all col of `values` with its associated string in `header`.
    Internal output is figure with 1 plot for all column , labelled by col name in Index.
    Optionally save as compressed numpy file with flags per viz (Recommended).
    Optionally save figure (Recommended) per viz. 
    """
    n_cols = len(metric_df.columns)
    fig, ax = plt.subplots(1, 1, sharex=True, sharey=True)
    for col in metric_df.columns:
        # display(list(metric_df.index.get_level_values(1)))
        # display(metric_df.loc[:, col].describe())
        # display(metric_df.loc[:, col].max())
        line, = ax.plot(list(metric_df.index.to_numpy(dtype=np.int32)), list(metric_df.loc[:, col].values))
        line.set_label(col)
    ax.set_title(f"{unified_name} Plot")

    ax.tick_params(axis='x')
    ax.tick_params(axis='y')

    ax.set_ylabel(unified_name)
    ax.set_ylabel(unified_name)

    ax.legend()
    plt.show()

In [ ]:
"""
Accuracy metric analysis helpers. 
"""
from typing import Literal

def agg_per_epoch(data_df: pd.DataFrame) -> pd.DataFrame:
    agg_df = data_df.groupby(level="epochs").mean(numeric_only=True)
    return agg_df

def agg_global_avg(data_df: pd.DataFrame) -> pd.DataFrame:
    agg_df = data_df.mean(axis=1).to_frame() # transpose replaces lack of axes
    agg_df.reset_index(drop=True, inplace=True) # strip the multi-index required mainly for per_epoch method but carried over here
    agg_df.columns.name = "global_avg"
    return agg_df

def summarize_runs_acc(summary_mode: Literal["per_epoch"] | Literal["global_avg"], run_metrics_paths: list[Path], prefix:str, custom_metric: Type[MyMultiClassAccuracy | GCH_Metric | RGCH_Metric]):
    """ 
    This summarises GC metrics across a run.
    Output is in the parent directory of the current path.  
    """
    enumerated_metrics = enumerate(run_metrics_paths)

    sum_df: pd.DataFrame = pd.DataFrame(np.arange(1))
    metadata: MetricMetadata = MetricMetadata(epochs=np.arange(1), steps=np.arange(1))
    metadata_df: pd.DataFrame = pd.DataFrame(np.arange(1))
    sum_sq_df: pd.DataFrame = pd.DataFrame(np.arange(1))
    idx: int = 0
    agg_df: pd.DataFrame = pd.DataFrame(np.arange(1))
    
    for idx, train_metrics_path in enumerated_metrics:
        metric = torch.load(train_metrics_path.resolve(), map_location="cpu")

        data_cols = [k for k in list(metric.keys()) if k != "epochs" and k != "steps"] # Do not add epochs, steps back into classes, they get merged into expected keys, and their state is reloaded when seen as part of labels, as key names, then again as metadata frmo load state dict method

        base_metric = custom_metric(data_cols) 
        base_metric.load_state_dict(metric)

        computed = base_metric.compute()

        data_df = pd.DataFrame(computed)
        data_df.replace(100, np.nan, inplace=True)
        data_df.replace(float('nan'), np.nan, inplace=True)
        data_df = data_df.map(lambda val: val.item() if isinstance(val, torch.Tensor) else val)
        
        if idx == 0:
            path_info = f"Showing run summary results in {train_metrics_path.resolve()}"
            display(path_info)

            # Convert tensors not addressed by `.compute()`
            clean_epochs = np.asarray(base_metric.metric_state["epochs"]) 
            clean_steps = np.asarray(base_metric.metric_state["steps"]) 

            metadata = MetricMetadata(epochs=clean_epochs, steps=clean_steps) # Unify values in new dict
            multiindex = pd.MultiIndex.from_arrays(
                [
                    list(metadata["epochs"]), 
                    list(metadata["steps"]) # list of lists format required
                ],
                names=["epochs", "steps"]
            )
            data_df.index = multiindex
            data_df.columns.name = "classes"
            # Across epoch aggregate - Get mean average per class 
            if summary_mode == "per_epoch":
                agg_df = agg_per_epoch(data_df=data_df)  
            elif summary_mode == "global_avg" :
                agg_df = agg_global_avg(data_df=data_df)
            else: 
                raise ValueError(f"{summary_mode} is not one of options")

            sum_df = pd.DataFrame(data=np.zeros(agg_df.shape), index=agg_df.index, columns=agg_df.columns)
            sum_sq_df = pd.DataFrame(data=np.zeros(agg_df.shape), index=agg_df.index, columns=agg_df.columns)
        
        sum_df = sum_df + agg_df 
        sum_sq_df = sum_sq_df + (agg_df ** 2) 

    num_metrics = idx + 1 
    global_mean_df = sum_df / (idx + 1) 
    global_std_df = (sum_sq_df / num_metrics) - (global_mean_df ** 2) 

    viz_all_col_unsized(
        metadata_df=metadata_df,
        metric_df=global_mean_df, 
        unified_name=f"{prefix} mean"
    )    
    viz_all_col_unsized(
        metadata_df=metadata_df,
        metric_df=global_std_df, 
        unified_name=f"{prefix} standard deviation"
    )

    # summary_mean = global_mean_df.describe()
    # summary_std = global_std_df.describe() 
    # display(summary_mean)
    # display(summary_std)
    # display(global_mean_df.head(10))
    # display(global_std_df.head(10))

In [ ]:
""" 
Main entry point for acc analysis. 
"""
def analyse_acc():
    for experim_dir in sorted(outputs_dir.iterdir()):
        version_dirs = sorted(experim_dir.glob("multijob/R*"))

        for v_dir in version_dirs:
            print("Train accuracy run metrics for", v_dir)

            # Get all csv file path matches to default logger file name for standard metrics
            train_acc_paths = sorted(v_dir.rglob("run*/train_acc.pt", case_sensitive=False))
            if len(train_acc_paths) > 0:
                summarize_runs_acc(summary_mode="per_epoch", run_metrics_paths= train_acc_paths, prefix=train_acc_paths[0].stem, custom_metric=MyMultiClassAccuracy)
                summarize_runs_acc(summary_mode="global_avg", run_metrics_paths= train_acc_paths, prefix=train_acc_paths[0].stem, custom_metric=MyMultiClassAccuracy)

            print("Val accuracy run metrics for", v_dir)

            val_acc_paths = sorted(v_dir.rglob("run*/val_acc.pt", case_sensitive=False))
            if len(val_acc_paths) > 0:
                summarize_runs_acc(summary_mode="per_epoch", run_metrics_paths= val_acc_paths, prefix=val_acc_paths[0].stem, custom_metric=MyMultiClassAccuracy)
                summarize_runs_acc(summary_mode="global_avg", run_metrics_paths= val_acc_paths, prefix=val_acc_paths[0].stem, custom_metric=MyMultiClassAccuracy)

In [ ]:
""" 
GCS analysis helpers. 
"""
def summarize_runs_gc(run_metrics_paths: list[Path], prefix:str, custom_metric: Type[MyMultiClassAccuracy | GCH_Metric | RGCH_Metric]):
    """ 
    This summarises GC metrics across a run.
    Output is in the parent directory of the current path.  
    """
    enumerated_metrics = enumerate(run_metrics_paths)

    sum_df: pd.DataFrame = pd.DataFrame(np.arange(1))
    metadata: MetricMetadata = MetricMetadata(epochs=np.arange(1), steps=np.arange(1))
    metadata_df: pd.DataFrame = pd.DataFrame(np.arange(1))
    sum_sq_df: pd.DataFrame = pd.DataFrame(np.arange(1))
    idx: int = 0
    
    for idx, train_metrics_path in enumerated_metrics:
        metric = torch.load(train_metrics_path.resolve(), map_location="cpu")

        data_cols = [k for k in list(metric.keys()) if k != "epochs" and k != "steps"] # Do not add epochs, steps back into classes, they get merged into expected keys, and their state is reloaded when seen as part of labels, as key names, then again as metadata frmo load state dict method

        base_metric = custom_metric(data_cols) 
        base_metric.load_state_dict(metric)

        computed = base_metric.compute()

        data_df = pd.DataFrame(computed)
        data_df.replace(100, np.nan, inplace=True)
        data_df.replace(float('nan'), np.nan, inplace=True)
        data_df = data_df.map(lambda val: val.item() if isinstance(val, torch.Tensor) else val)
        
        if idx == 0:
            path_info = f"Showing run summary results in {train_metrics_path.resolve()}"
            display(path_info)

            # Convert tensors not addressed by `.compute()`
            clean_epochs = np.asarray(base_metric.metric_state["epochs"]) 
            clean_steps = np.asarray(base_metric.metric_state["steps"]) 

            metadata = MetricMetadata(epochs=clean_epochs, steps=clean_steps) # Unify values in new dict
            metadata_df = pd.DataFrame(metadata)
            sum_df = pd.DataFrame(data=np.zeros(data_df.shape), index=data_df.index, columns=data_df.columns)
            sum_sq_df = pd.DataFrame(data=np.zeros(data_df.shape), index=data_df.index, columns=data_df.columns)
        
        sum_df = sum_df + data_df 
        sum_sq_df = sum_sq_df + (data_df ** 2) 

    num_metrics = idx + 1 
    global_mean_df = sum_df / (idx + 1) 
    global_std_df = (sum_sq_df / num_metrics) - (global_mean_df ** 2) 

    viz_all_col_unsized(
        metadata_df=metadata_df,
        metric_df=global_mean_df, 
        unified_name=f"{prefix} mean"
    )    
    viz_all_col_unsized(
        metadata_df=metadata_df,
        metric_df=global_std_df, 
        unified_name=f"{prefix} standard deviation"
    )

    # summary_mean = global_mean_df.describe()
    # summary_std = global_std_df.describe() 
    # display(summary_mean)
    # display(summary_std)
    # display(global_mean_df.head(10))
    # display(global_std_df.head(10))

In [ ]:
""" 
Main entry point for GC analysis. 
"""
def analyse_gc():
    for experim_dir in sorted(outputs_dir.iterdir()):
        version_dirs = sorted(experim_dir.glob("multijob/R*"))

        for v_dir in version_dirs:
            print("Train accuracy run metrics for", v_dir)

            print("Val accuracy run metrics for", v_dir)

            # Get all csv file path matches to custom name for both cs metrics
            gcs_paths = sorted(v_dir.rglob(f"run*/train_{corr_store_names["normed"]}.pt", case_sensitive=False))
            if len(gcs_paths) > 0:
                summarize_runs_gc(gcs_paths, "gcs", GCH_Metric)

            rgcs_paths = sorted(v_dir.rglob(f"run*/train_{corr_store_names["raw"]}.pt", case_sensitive=False))
            if len(rgcs_paths) > 0:
                summarize_runs_gc(rgcs_paths, "rgcs", RGCH_Metric)

In [ ]:
analyse_acc()
analyse_gc()